# 12 — Results and paper integration

## Question

How do the real-data artifacts populate the PENDING placeholders in `docs/STAGE_10_PAPER.md` §7?

## Why this test exists

The paper manuscript has 9 PENDING subsections in §7 (real-data experiment). Each PENDING placeholder is filled from a real_* artifact by tracing the artifact's field to the paper's claim.

## Method

Trace each §7.x claim to its source artifact. Report the values. Apply the case-label (A / B / C) and the discussion framework. Do not invent values. Do not re-tune.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.

**No quantum-advantage claim.** This work does NOT claim QAOA outperforms classical optimization. The 11-qubit instance is small enough that exact classical optimization is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. Any quantum-advantage language is explicitly avoided.


## Implementation


In [ ]:
import json, pathlib
print('Populating §7 PENDING placeholders from real_* artifacts:')
print()

# §7.1: real-data acquisition
raw = pathlib.Path('../artifacts/real_raw_sessions.json')
if raw.exists():
    print(f'§7.1: real_data acquired = True (manifest sha256: {json.loads(raw.read_text())["sha256"][:16]}...)')

# §7.2: empirical uncertainty
us = pathlib.Path('../artifacts/real_uncertainty_statistics.json')
if us.exists():
    s = json.loads(us.read_text())
    print(f'§7.2: real ΔE mean = {s["calibration"]["Delta_e_distribution"]["mean"]:.4f} kWh')
    print(f'§7.2: real Δd mean = {s["calibration"]["Delta_d_distribution"]["mean"]:.2f} min')

# §7.3: real scenario distribution
sc = pathlib.Path('../artifacts/real_scenarios_K8.json')
if sc.exists():
    s = json.loads(sc.read_text())
    print(f'§7.3: K = {s["K"]}; seed = {s["seed"]}; n_clusters = {len(s["scenarios"])}')

# §7.4: real γ
cp = pathlib.Path('../artifacts/real_calibration_parameters.json')
if cp.exists():
    s = json.loads(cp.read_text())
    print(f'§7.4: real γ = {s["gamma"]:.4f}; ρ_d_robust = {s["rho_d_robust"]:.4f}')

# §7.5: F0/F1/F2/F3
for name, label in [('real_f0_results','§7.5 F0'), ('real_f1_robust_results','§7.5 F1'),
                     ('real_f2_adopt_results','§7.5 F2'), ('real_f3_oracle_results','§7.5 F3')]:
    p = pathlib.Path(f'../artifacts/{name}.json')
    if p.exists():
        s = json.loads(p.read_text())
        print(f'{label}: classical_optimum = {s.get("classical_optimum"):.4f}')

# §7.6: held-out P(feasible)
ho = pathlib.Path('../artifacts/real_heldout_results.json')
if ho.exists():
    s = json.loads(ho.read_text())
    for fname, h in s.items():
        print(f'§7.6 {fname} P(feas) = {h["P_feasible"]:.4f}')

# §7.7: paired statistics
ps = pathlib.Path('../artifacts/real_paired_statistics.json')
if ps.exists():
    s = json.loads(ps.read_text())
    prim = s['primary_F2_vs_F0']
    print(f'§7.7: F2 vs F0 P(feas) diff = {prim["diff_mean"]:.4f}; 95% CI = [{prim["ci_lo"]:.4f}, {prim["ci_hi"]:.4f}]')

# §7.8: distribution shift
ds = pathlib.Path('../artifacts/real_distribution_shift.json')
if ds.exists():
    s = json.loads(ds.read_text())
    print(f'§7.8: case_label = {s["case_label"]}')


## Discussion framework (Case A / B / C)

The discussion applies the case-label honestly:

- **Case A (mild shift)**: F2 is expected to behave similarly to F0   on the held-out set, and the bootstrap CI is expected to include   0. The paper reports this as a negative result: ADOPT does not   help when the distribution does not shift.
- **Case B (moderate shift)**: F2 is expected to dominate F0 on   P(feasible) by a small but statistically significant margin. The   paper reports this as a positive result for ADOPT under moderate   drift.
- **Case C (severe shift)**: F2 may not generalize; the frozen γ is   calibrated for the calibration distribution. The paper reports   the result as-is and discusses the limitation.

**No case is preferred.** The paper's contribution is the methodology, not the verdict. The verdict is whatever the data say.


## Limitations

- The 11-qubit instance is small; the result does not predict   behavior on larger instances.
- The Caltech site is one site; the result may not generalize to   JPL, Caltech Office, or other ACN-Data sites.
- The 6-month held-out window is short; longer windows would   give more robust deployment estimates.
- The smooth two-sided site-cap form (Stage 4 Part A) is a   modeling choice; the one-sided operational form would give   different schedules (see Stage 6 Part K for the trade-off).
- ADOPT is a single-scalar mechanism; it cannot represent full   distributional change.
- The frozen QAOA configuration is one of many; p=2 and other   configurations are sensitivity checks.
- The headline result is F2 vs F0 P(feasible); cost, unmet, peak,   deadline, and site are secondary outcomes, with Bonferroni   correction on the P(feas) comparisons only.

## Reproducibility

- **Code**: `stage3/`, `stage4/`, `stage5/`, `stage6/`, `stage7/`,   `stage8/`, `stage9/` (Python 3.13).
- **Data**: live ACN-Data Caltech sessions, 2018-05-01 to   2020-01-01 UTC, via `https://ev.caltech.edu/api/v1/sessions/  caltech` with `ACN_API_TOKEN` env var.
- **Frozen config**: `artifacts/final_experiment_config.json`   (version `stage7.v1`); raw-file SHA-256   `4a08e1e65587cc904521ff1bfc955b30671a5cb3485664d8b75f1ad93d9013b3`.
- **Driver**: `python -m stage9.real_experiment` from the repo root.


## Result

See the code outputs above for the numerical results. The interpretation is in the next section.


## Interpretation

See the Limitations section below for what the result does NOT show.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Result

See the code outputs above for the numerical results. The interpretation is in the next section.


## Interpretation

See the Limitations section below for what the result does NOT show.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.
